In [4]:
import pandas as pd
from tqdm.auto import tqdm
import glob
from project_package import data_collection as dc
from project_package import utility

In [16]:
#load dataset form csv
df_ncr = utility.read_data('ncr_ride_bookings(with_loc)')
print(df_ncr.shape)
# Combine Date + Time into a single datetime
df_ncr['datetime'] = pd.to_datetime(df_ncr['Date'] + " " + df_ncr['Time'])

# Extract date and hour for merging
df_ncr['date'] = df_ncr['datetime'].dt.date
df_ncr['hour'] = df_ncr['datetime'].dt.hour
df_ncr.head()

(150000, 31)


,Date,Time,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,...,pick_region,pick_locality,drop_longitude,drop_latitude,drop_address,drop_region,drop_locality,datetime,date,hour
0,2024-03-23,12:29:38,"""CNR5884300""",No Driver Found,"""CID1982111""",eBike,Palam Vihar,Jhilmil,NaN,NaN,...,Delhi,Delhi,77.311751,28.670789,"DGD Jhilmil, Delhi, India",Delhi,Delhi,2024-03-23 12:29:38,2024-03-23,12
1,2024-11-29,18:01:39,"""CNR1326809""",Incomplete,"""CID4604802""",Go Sedan,Shastri Nagar,Gurgaon Sector 56,4.9,14.0,...,Maharashtra,Pune,77.011193,28.489101,"Gurgaon, Gurugram, HR, India",Haryana,Gurugram,2024-11-29 18:01:39,2024-11-29,18
2,2024-08-23,08:56:10,"""CNR8494506""",Completed,"""CID9202816""",Auto,Khandsa,Malviya Nagar,13.4,25.8,...,Uttar Pradesh,Khandsara,77.212385,28.534341,"Malviya Nagar, Delhi, India",Delhi,Delhi,2024-08-23 08:56:10,2024-08-23,8
3,2024-10-21,17:17:25,"""CNR8906825""",Completed,"""CID2610914""",Premier Sedan,Central Secretariat,Inderlok,13.1,28.5,...,Karnataka,Mangalore,77.167640,28.672408,"DGD, Inderlok, Delhi, India",Delhi,Delhi,2024-10-21 17:17:25,2024-10-21,17
4,2024-09-16,22:08:00,"""CNR1950162""",Completed,"""CID9933542""",Bike,Ghitorni Village,Khan Market,5.3,19.6,...,Odisha,NaN,77.229128,28.602245,"Khan Market, Delhi, India",Delhi,Delhi,2024-09-16 22:08:00,2024-09-16,22


In [18]:
df_ncr['hour'].unique()

array([12, 18,  8, 17, 22,  9, 15, 19, 16, 10, 21,  6, 11, 20,  5, 14, 13,
        3,  7,  0,  4,  2,  1, 23], dtype=int32)

In [10]:
# Grab all weather files
files = sorted(glob.glob("datasets/raw/weather/*.csv"))
print("Files found:", files)

# Read and concatenate
df_weather = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
df_weather['time'] = pd.to_datetime(df_weather['time'])

# Extract date and hour
df_weather['date'] = df_weather['time'].dt.date
df_weather['hour'] = df_weather['time'].dt.hour

print(df_weather.shape)

df_weather.head()

Files found: ['datasets/raw/weather/weather_data0.csv', 'datasets/raw/weather/weather_data1.csv', 'datasets/raw/weather/weather_data2.csv', 'datasets/raw/weather/weather_data3.csv', 'datasets/raw/weather/weather_data4.csv', 'datasets/raw/weather/weather_data5.csv']
(4878509, 12)


,time,address,temperature_2m,relative_humidity_2m,dew_point_2m,apparent_temperature,precipitation,rain,snowfall,wind_speed_10m,date,hour
0,2024-01-01 00:00:00,"kuruva palam, KL, India",24.2,82,20.9,27.8,0.0,0.0,0.0,5.2,2024-01-01,0
1,2024-01-01 01:00:00,"kuruva palam, KL, India",24.8,75,20.1,27.7,0.0,0.0,0.0,7.1,2024-01-01,1
2,2024-01-01 02:00:00,"kuruva palam, KL, India",25.2,69,19.0,27.1,0.0,0.0,0.0,10.3,2024-01-01,2
3,2024-01-01 03:00:00,"kuruva palam, KL, India",26.9,60,18.4,28.1,0.0,0.0,0.0,13.1,2024-01-01,3
4,2024-01-01 04:00:00,"kuruva palam, KL, India",29.2,52,18.4,30.4,0.0,0.0,0.0,12.6,2024-01-01,4


In [24]:
df_weather_unique = df_weather.drop_duplicates(subset=["address", "date", "hour"])
df_weather_unique.shape

(4629168, 12)

In [28]:
# merge weather data with ncr ride bookings

def merge_weather(
    df_main, 
    df_weather, 
    main_key, 
    weather_key="address", 
    time_keys_main=["date", "hour"], 
    time_keys_weather=["date", "hour"], 
    suffix="_pickup"
):
    """
    Merge weather data onto a main DataFrame by matching address + date + hour.
    -------
    pd.DataFrame
        Merged DataFrame with weather columns suffixed.
    """
    # Columns to keep without renaming (join keys)
    join_keys = [weather_key] + time_keys_weather

    # Rename weather columns except join keys
    weather_renamed = df_weather.rename(
        columns={col: f"{col}{suffix}" for col in df_weather.columns if col not in join_keys}
    )

    # Merge on address + date + hour
    df_merged = df_main.merge(
        weather_renamed,
        left_on=[main_key] + time_keys_main,
        right_on=join_keys,
        how="left"
    )

    # Drop duplicate join columns from weather
    #df_merged = df_merged.drop(columns=join_keys)
    return df_merged

In [29]:
df_merged_pickup = merge_weather(df_ncr, df_weather_unique, main_key="pick_address", suffix="_pickup")

In [30]:
df_merged_pickup.shape

(150000, 44)

In [31]:
df_merged = merge_weather(df_merged_pickup,df_weather_unique,main_key = 'drop_address', suffix = '_drop')

In [33]:
df_merged.shape

(150000, 54)

In [35]:
utility.save_dataframe(df_merged,file_name='ncr_weather_merged_data',directory='datasets/raw/merged')

Save data completed.
